# Dogs vs Cats
This notebook runs Part 1: baseline Dogs vs Cats classifiers under identity and fixed tile-wise permutations.


## Setup

### Global / External Imports
Import third-party libraries used only for notebook orchestration and display.


most basic imports

In [ ]:
import sys, os
from pathlib import Path
import subprocess

make sure installations are made before other imports

In [ ]:
def get_project_root():
    project_root = Path.cwd()
    if project_root.name == 'notebooks':
        project_root = project_root.parents[1]
    elif project_root.name == 'src':
        project_root = project_root.parent
    print(f"Project root: {project_root}")
    return project_root

In [ ]:
REPO_PATH = get_project_root()

### Local Imports
Import project modules. Part 1 orchestration lives in this notebook; reusable training, preprocessing, and evaluation helpers stay in `src`.


before local iports make sure repo is updated

In [ ]:
def update_git_repo():
    try:
        # הרצת הפקודה ובדיקה אם היא הצליחה (check=True)
        result = subprocess.run(['git', 'pull'], check=True, capture_output=True, text=True)
        print("Update successful:", result.stdout)
    except subprocess.CalledProcessError as e:
        # אם הפקודה נכשלה, המערכת תדפיס שגיאה ותעצור את ריצת התאים הבאים
        print("Git pull failed!")
        print("Error details:", e.stderr)
        raise Exception("Stopping execution due to Git Pull failure")

In [ ]:
update_git_repo()


make local imports

In [ ]:
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

from src.evaluation.experiment_results import (
    experiment_output_paths,
    get_device,
    load_experiment_samples,
    plot_accuracy_vs_tiles,
    save_aggregated_accuracy,
    save_rows,
)
from src.preprocessing.dogs_cats import build_dataloaders, class_counts
from src.preprocessing.permutations import build_permutation_records
from src.training.experiment_steps import train_model_configuration
from src.utils.io import ensure_dir, save_csv
from src.utils.reproducibility import seed_everything


### Setup configs

In [ ]:
from dataclasses import dataclass, field
from typing import Literal


@dataclass
class CVExperimentConfig:
    # General experiment configuration
    part: str = "part1"
    config_name: str = "part1_baselines"
    device: str = "auto"
    seeds: list[int] = field(default_factory=lambda: [0])
    deterministic: bool = False

    # Directory configuration
    data_dir: str = "/Users/royrubin/Documents/GitHub/MLDS_Final_Project/data/dogs-vs-cats/train"
    outputs_dir: str = "/Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs"
    results_dir: str = "/Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/results"
    figures_dir: str = "/Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/figures"

    # Dataset splitting configuration
    sample_data: bool = False
    sample_limit: int = 256
    val_fraction: float = 0.2
    test_fraction: float = 0.0

    # Image preprocessing configuration
    image_size: int = 224

    # Data loading configuration
    batch_size: int = 16
    num_workers: int = 2

    # Model configuration
    num_classes: int = 2
    model_names: list[str] = field(
        default_factory=lambda: ["resnet18", "swin_t", "convmixer"]
    )
    pretrained: bool = False

    # ConvMixer-specific configuration
    convmixer_dim: int = 128
    convmixer_depth: int = 4

    # Grid and permutation experiment configuration
    grid_sizes: list[int] = field(default_factory=lambda: [1, 2, 3, 4])
    num_permutations: int = 2
    permutation_seed: int = 42

    # Training configuration
    epochs: int = 1
    learning_rate: float = 0.0003
    optimizer: Literal["adamw"] = "adamw"
    weight_decay: float = 0.0001
    use_amp: bool = False

    # reproduction configuration
    seed: int = 42
    max_threads: int = 4

    # environment configuration
    using_google_colab: bool = False


In [ ]:
configs = CVExperimentConfig()
configs


### Global Definitions
Define paths and load the grouped YAML config.


### setup paths in relation to usa

In [ ]:
## Code to move to utils
def is_code_running_on_colab():
    try:
        from google.colab import drive
        return True
    except ImportError:
        return False
    # return 'google.colab' in sys.modules


In [ ]:
if is_code_running_on_colab():
    print("Note: running on Google Colab")
    configs.using_google_colab = True

    # mount google drive
    from google.colab import drive
    drive.mount('/content/drive')

    configs.data_dir = "/content/drive/MyDrive/MLDS_Final_Project/data"       # <-- change
    configs.outputs_dir = "/content/drive/MyDrive/MLDS_Final_Project/outputs" # <-- change

    # Make relevant instalation only if using collab (otherwise, already installed)
    os.chdir(REPO_PATH)
    %pip install -r requirements.txt
    

    # Sanity check
    assert os.path.exists(configs.data_dir), f"Data dir {configs.data_dir} does not exist"
    os.makedirs(configs.outputs_dir, exist_ok=True)
    print("CWD:", os.getcwd())
    print("Data dir:", configs.data_dir)
    print("Output dir:", configs.outputs_dir)

else:
    print("Note: not running on Google Colab, using local paths")

    configs.sample_data = True

final imports after knowing repo path

In [ ]:
from IPython.display import Image, display
import pandas as pd
import random, numpy as np
import torch, torchvision

## Experiments

In [ ]:
dont forget to sample if config says so

In [ ]:
import json


def as_config_dict(configs):
    """Return the notebook dataclass config as a plain dictionary."""
    config = dict(vars(configs))
    return config


def part1_output_paths(config):
    """Build stable Part 1 output paths for notebook display."""
    paths = experiment_output_paths(config['results_dir'], config['figures_dir'], 'part1')
    paths['accuracy_plot'] = paths['figure']
    return paths


def load_part1_data(config, seed=None):
    """Load the Dogs vs Cats train/validation/test split for Part 1."""
    selected_seed = int(config.get('seeds', [0])[0] if seed is None else seed)
    samples = load_experiment_samples(config, seed=selected_seed)
    return samples


def build_part1_result_row(config, run_id, model_name, record, seed, metrics):
    """Create one raw Part 1 result row."""
    row = {
        'part': 'part1',
        'run_id': run_id,
        'config_name': config.get('config_name', 'part1_baselines'),
        'model_name': model_name,
        'grid_size': record.grid_size,
        'num_tiles': record.grid_size * record.grid_size,
        'permutation_id': record.permutation_id,
        'permutation_seed': record.permutation_seed,
        'seed': int(seed),
        **metrics,
    }
    return row


def run_part1_notebook(config):
    """Run Part 1 from this notebook and save raw, aggregated, and figure outputs."""
    ensure_dir(config['results_dir'])
    ensure_dir(config['figures_dir'])
    device = get_device(config)
    output_paths = part1_output_paths(config)
    run_id = config.get('run_id', 'part1_notebook')
    rows = []

    permutation_records = build_permutation_records(
        grid_sizes=[int(value) for value in config.get('grid_sizes', [1, 2, 3, 4])],
        num_permutations=int(config.get('num_permutations', 2)),
        permutation_seed=int(config.get('permutation_seed', 42)),
        include_identity=True,
    )
    permutation_rows = [record.__dict__ | {'permutation': json.dumps(record.permutation)} for record in permutation_records]
    save_csv(permutation_rows, output_paths['permutations'])

    for seed in config.get('seeds', [0]):
        seed_everything(int(seed), deterministic=bool(config.get('deterministic', False)))
        train_samples, validation_samples, _ = load_part1_data(config, seed=int(seed))
        for model_name in config.get('model_names', ['resnet18', 'swin_t', 'convmixer']):
            for record in permutation_records:
                if record.grid_size == 1 and record.permutation_id > 0:
                    continue
                train_loader, validation_loader = build_dataloaders(
                    train_samples,
                    validation_samples,
                    image_size=int(config.get('image_size', 224)),
                    grid_size=record.grid_size,
                    permutation=record.permutation,
                    seed=int(seed),
                    batch_size=int(config.get('batch_size', 32)),
                    num_workers=int(config.get('num_workers', 2)),
                    standard_augmentation=False,
                )
                metrics = train_model_configuration(
                    config,
                    model_name,
                    train_loader,
                    validation_loader,
                    device,
                    overrides={'pretrained': bool(config.get('pretrained', False)) and model_name != 'convmixer'},
                )
                row = build_part1_result_row(config, run_id, model_name, record, seed, metrics)
                rows.append(row)
                save_rows(rows, output_paths['raw_results'])

    raw_results = pd.DataFrame(rows)
    aggregated_results = save_aggregated_accuracy(
        raw_results,
        ['model_name', 'grid_size', 'num_tiles'],
        output_paths['aggregated_results'],
    )
    plot_accuracy_vs_tiles(aggregated_results, output_paths['accuracy_plot'])
    return aggregated_results


### Setup Exp

In [ ]:
# 4. Reproducibility (important for multiple notebooks
random.seed(configs.seed)
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
torch.set_num_threads(configs.max_threads)  # avoid contention across notebooks

In [ ]:
part1_config = as_config_dict(configs)
output_paths = part1_output_paths(part1_config)
output_paths


### Data Loading
Discover the configured Dogs vs Cats split before training.


In [ ]:
train_samples, validation_samples, test_samples = load_part1_data(part1_config)
print(f'Train samples: {len(train_samples)}')
print(f'Validation samples: {len(validation_samples)}')
print(f'Test samples: {len(test_samples)}')
print('Train class counts:', class_counts(train_samples))


### Experiments - Run Baselines
Run the configured baseline grid/model/permutation sweep. Re-run this cell to regenerate Part 1 outputs.


In [ ]:
aggregated_results = run_part1_notebook(part1_config)
display(aggregated_results)


### Experiments - Results Table
Reload the saved aggregated CSV so the report is reproducible from disk.


In [ ]:
saved_results = {
    'raw': pd.read_csv(output_paths['raw_results']),
    'aggregated': pd.read_csv(output_paths['aggregated_results']),
}
display(saved_results['aggregated'])


### Experiments - Accuracy Plot
Display the saved accuracy-vs-number-of-tiles plot.


In [ ]:
figure_path = output_paths['accuracy_plot']
if Path(figure_path).exists():
    display(Image(filename=figure_path))
else:
    print(f'Plot not found yet: {figure_path}')
